# LRO Staging & Normalized Data Exploration

This notebook connects to the PostgreSQL database and allows you to explore the staging and normalized tables.

In [ ]:
# Install dependencies if needed
# !pip install pandas psycopg2-binary sqlalchemy

In [5]:
import pandas as pd
from sqlalchemy import create_engine

# Database connection settings (matching docker-compose.yml)
DB_CONFIG = {
    "host": "localhost",
    "port": 5433,
    "dbname": "database",
    "user": "admin",
    "password": "password",
}

# Create connection string
db_url = f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"

# Create engine
engine = create_engine(db_url)

print("Connection engine created successfully.")

Connection engine created successfully.


## 1. Metadata Tables (Full Content)
Printing all normalized tables except the large observation data.

In [6]:
metadata_tables = ['owner', 'site', 'unit', 'method', 'variable', 'processing_level', 'qualifier']

for table in metadata_tables:
    print(f"\n--- Table: {table.upper()} ---")
    try:
        display(pd.read_sql(f"SELECT * FROM {table}", engine))
    except Exception as e:
        print(f"Could not read table {table}: {e}")


--- Table: OWNER ---


,owner_id,name,owner,contact_email
0,1,Jeff Horsburgh,Jeff Horsburgh,jeff.horsburgh@usu.edu



--- Table: SITE ---


,site_id,site_code,name,description,site_type,latitude,longitude,elevation_m,state,county
0,1,LR_TG_C,Climate Station at Tony Grove,Logan River Observatory climate monitoring sta...,Atmosphere,41.885493,-111.568767,1927.86,Utah,Cache
1,2,LR_GC_C,Climate Station at Logan River Golf Course,Logan River Observatory climate monitoring sta...,Atmosphere,41.705643,-111.854268,1364.89,Utah,Cache
2,3,LR_WaterLab_AA,Logan River at the Utah Water Research Laborat...,Logan River Observatory monitoring site for Lo...,Stream,41.739034,-111.795742,1414.00,Utah,Cache
3,4,LR_Mendon_AA,Logan River at Mendon Road (600 South),Logan River Observatory monitoring site for Lo...,Stream,41.720533,-111.886928,1353.50,Utah,Cache



--- Table: UNIT ---


...

,variable_id,name,definition,description,variable_type,variable_code,unit_id,method_id
0,1,Snow depth,Snow depth is the vertical distance from the g...,https://en.wikipedia.org/wiki/Vertical_position,Hydrology,SnowDepth,1,1
1,2,Water Temperature,Water temperature is the measure of the therma...,https://en.wikipedia.org/wiki/Temperature,Water Quality,WaterTemp,2,2
2,3,"Oxygen, dissolved",Dissolved oxygen is the concentration of oxyge...,https://en.wikipedia.org/wiki/Oxygen_saturation,Water Quality,ODO,3,6
3,4,Discharge,Discharge is the volumetric flow rate of water...,https://en.wikipedia.org/wiki/Discharge_(hydro...,Hydrology,Discharge,4,4
4,5,Air Temperature,Air temperature is the measure of the average ...,https://en.wikipedia.org/wiki/Temperature,Climate,AirTemp,2,5
5,6,Precipitation,"Precipitation is any form of water, such as ra...",https://en.wikipedia.org/wiki/Precipitation,Hydrology,Precip,1,3



--- Table: PROCESSING_LEVEL ---


,processing_level_id,code,definition,explanation
0,1,1.5,Provisional data,Quality controlled data that is subject to cha...
1,2,1,Quality controlled data,Quality controlled data that have passed quali...
2,3,0,Raw data,Raw and unprocessed data and data products tha...



--- Table: QUALIFIER ---


,qualifier_id,qualifier_code,description
0,1,LI,None
1,2,S,None


## 2. Observation Data (Samples)
Printing row count and first 10 rows from the large `datastream` table.

In [7]:
total_rows = pd.read_sql("SELECT COUNT(*) as total FROM datastream", engine).iloc[0,0]
print(f"Total rows in DATASTREAM: {total_rows:,}")

print("\n--- Table: DATASTREAM (First 10 Rows) ---")
pd.read_sql("SELECT * FROM datastream LIMIT 10", engine)

Total rows in DATASTREAM: 4,067,863

--- Table: DATASTREAM (First 10 Rows) ---


,datastream_id,datetime_utc,value,site_id,variable_id,owner_id,qualifier_id,processing_level_id
0,1,2014-01-01 07:15:00,144.797849,4,4,1,None,1
1,2,2014-01-01 07:30:00,144.604809,4,4,1,None,1
2,3,2014-01-01 07:45:00,145.136261,4,4,1,None,1
3,4,2014-01-01 08:00:00,144.604809,4,4,1,None,1
4,5,2014-01-01 08:15:00,145.039495,4,4,1,None,1
5,6,2014-01-01 08:30:00,145.184668,4,4,1,None,1
6,7,2014-01-01 08:45:00,145.329979,4,4,1,None,1
7,8,2014-01-01 09:00:00,145.184668,4,4,1,None,1
8,9,2014-01-01 09:15:00,145.378447,4,4,1,None,1
9,10,2014-01-01 09:30:00,145.329979,4,4,1,None,1


## 3. Staging Table (Legacy)
The raw ingested data before normalization.

In [9]:
print(f"Total rows in staging table: {pd.read_sql('SELECT COUNT(*) FROM staging', engine).iloc[0,0]:,}")
pd.read_sql("SELECT * FROM staging LIMIT 5", engine)

Total rows in staging table: 0


,staging_id,workspace_name,owner_name,contact_email,site_code,site_name,site_description,sampling_feature_type,site_type,latitude,...,intended_time_spacing_unit,aggregation_statistic,time_aggregation_interval,time_aggregation_interval_unit,qualifier_code,qualifier_description,datetime_utc,value,source_file,loaded_at
